# Figure: Solubilities during degassing for Fuego

In [ ]:
from pathlib import Path
import numpy as np

results_directory = Path().resolve().parent / "Model_Outputs"
SAVE_FIG = True

## Import data and styling

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from helpers.plot_styles import (
    PLOTLY_TICK_LEN,
    PLOTLY_FONT,
    PLOTLY_TICK_FONTSIZE,
    SAMPLE_DISPLAY_NAMES,
    TOOL_COLORS_HEX,
    TOOL_LINE_STYLE,
)
from helpers.degassing_data import load_all_systems

# --- USER INPUTS --- #
SAMPLES = ["MORB", "Kilauea", "Fuego", "Fogo"]
TOOLS  = ["DCompress", "DCompress (IM)", "EVo", "MAGEC", "SulfurX", "VolFe", "VESIcal_Iacono"]

In [ ]:
systems = load_all_systems(SAMPLES, TOOLS, results_dir=results_directory)

## Build the figure

In [ ]:
# Row definitions: (DataFrame column, y-axis label).
Y_ROWS = [
    ("H2O","log<sub>10</sub>H' - H<sub>2</sub>O",-2.5,-1.8),
    ("CO2", "log<sub>10</sub>H'- CO<sub>2</sub>",-0.5,0.5),
    ("S2m","log<sub>10</sub>H' - S<sup>red</sup>",-14,-10),
    ("S6p","log<sub>10</sub>H' - S<sup>ox</sup>",5,9)
]

n_rows, n_cols = 1, len(Y_ROWS)
top_titles = [SAMPLE_DISPLAY_NAMES.get(s, s) for s in SAMPLES]
subplot_titles = top_titles + [""] * ((n_rows - 1) * n_cols)

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    shared_xaxes=False, vertical_spacing=0.03, horizontal_spacing=0.07,
)

r=1
sample = 'Fuego'
for c, (species,y_label,min,max) in enumerate(Y_ROWS, start=1):
    for tool in TOOLS:
        df = systems.get(sample, {}).get(tool)
        if df is None or "P_bars" not in df.columns:
            continue
        p_init = df["P_bars"].iloc[0]
        if p_init == 0:
            continue
        x_norm = df["P_bars"]# / p_init
        if species == 'H2O':
            solubility = np.log10((df['H2OT_m_wtpc']**2.)/(df['H2O_v_mf']*df['P_bars']))
        elif species == 'CO2':
            solubility = np.log10((df["CO2T_m_ppmw"])/(df['CO2_v_mf']*df['P_bars']))
        elif species == 'S2m':
            solubility = np.log10((df['ST_m_ppmw']*(1.-df['S6St_m'])*((10.**df['logfO2'])**1.5/(df['SO2_v_mf']*df['P_bars']))))
        elif species == 'S6p':
            solubility = np.log10((df['ST_m_ppmw']*df['S6St_m'])/(((10.**df['logfO2'])**0.5)*(df['SO2_v_mf']*df['P_bars'])))
        fig.add_trace(
            go.Scatter(
                mode="lines",
                x=x_norm, y=solubility,
                line=dict(color=TOOL_COLORS_HEX.get(tool, "#333"), width=2,
                              dash=TOOL_LINE_STYLE.get(tool, "solid"),
                              ),
            ),
            row=r, col=c,
        )
    fig.update_yaxes(title_text=y_label, range=[min,max], row=r, col=c, title_standoff=5)
    fig.update_xaxes(title_text="Pressure (bar)", row=r, col=c, range=[0, None], title_standoff=5)

fig.update_layout(
    height=290, width=1100,
    plot_bgcolor="white",
    margin=dict(t=10, r=10, l=10, b=10),
    font=PLOTLY_FONT,
    showlegend=False,
)
fig.update_xaxes(
    showline=True, linewidth=1, linecolor="black", mirror=True,
    ticks="outside", ticklen=PLOTLY_TICK_LEN, tickcolor="black",
    tickfont=dict(size=PLOTLY_TICK_FONTSIZE),
)
fig.update_yaxes(
    showline=True, linewidth=1, linecolor="black", mirror=True,
    ticks="outside", ticklen=PLOTLY_TICK_LEN, tickcolor="black",
    tickfont=dict(size=PLOTLY_TICK_FONTSIZE),
    rangemode="tozero",
)

if SAVE_FIG:
    fig.write_image("figures/Fig_solubilities_Fuego.png", scale=2, height=290, width=1100)

fig.show()